In [4]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split, WeightedRandomSampler
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm
import csv
from scipy.sparse import issparse
from model.sparsemax import Sparsemax


from model.model import MIL, EarlyStopping
from model.dataset import BagsDataset, custom_collate_fn

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#adata_1 = sc.read_h5ad('../test.h5ad')
#adata_2 = sc.read_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE/HumanLungCancerFFPE.h5ad')


In [ ]:
"""adata_radius_input = [
    (adata_1, 200, 'low'),
    (adata_2, 200, 'low'),
]"""

"adata_radius_input = [\n    (adata_1, 200, 'low'),\n    (adata_2, 200, 'low'),\n]"

In [2]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanOvarianCancer/T_cell.h5ad')

In [6]:
dataset = BagsDataset(adata, immune_cell='tcell', max_instances=None, radius=200, resolution='low', n_genes=500, k=2)
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

['0', '1']
Categories (2, object): ['0', '1']
Tumor cells shape after filtering: (1226, 17943)
Selecting top 500 genes based on mean expression
Top 500 genes: Index(['IGHM', 'IGLC1', 'IGKC', 'JCHAIN', 'B2M', 'TMSB4X', 'UBC', 'UBA52',
       'ACTB', 'FTL',
       ...
       'QKI', 'SGPL1', 'SLC38A2', 'YWHAQ', 'GBP5', 'CTNNB1', 'PTGS1', 'CLTC',
       'HP1BP3', 'DDIT4'],
      dtype='object', length=500)
Percentile value: 4.037618160247803
adata.obs[T] after binarization: AAACAAGTATCTCCCA-1    0
AAACAATCTACTAGCA-1    0
AAACACCAATAACTGC-1    1
AAACAGAGCGACTCCT-1    1
AAACAGCTTTCAGAAG-1    0
Name: T, dtype: int64
Preprocessed data: (3455, 500)


Creating Bags with radius 200: 100%|█████████████████████████| 3455/3455 [00:00<00:00, 21195.10it/s]

Total batches created: 613


In [7]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)



In [ ]:
distances, gene_expressions, labels, core_idxs, gene_names, cell_ids = next(iter(dataloader))


In [14]:
distances[0]

tensor([[200.0000],
        [141.4214],
        [200.0000],
        [141.4214],
        [141.4214],
        [  0.0000],
        [200.0000]])

In [15]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(-4.0),requires_grad=True)
        self.sparsemax = Sparsemax(dim=0)
        self.softmax = nn.Softmax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = self.a
        x = self.softmax(-torch.exp(a) * x)
        return x

In [16]:
model = Distance()
output = model(distances[1])
print(output)

tensor([[0.0218],
        [0.0218],
        [0.0218],
        [0.0218],
        [0.8492],
        [0.0637]], grad_fn=<SoftmaxBackward0>)


In [17]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x

In [18]:
gene_expressions[0]

tensor([[5.9022, 5.5666, 4.1879,  ..., 1.3946, 1.1603, 1.3519],
        [5.0775, 4.6695, 4.6598,  ..., 1.2014, 0.8292, 1.0325],
        [5.7203, 4.9806, 4.4486,  ..., 1.4324, 1.7079, 1.7079],
        ...,
        [5.1867, 4.7709, 4.2782,  ..., 0.8491, 1.4687, 1.4160],
        [5.6708, 5.1197, 4.5867,  ..., 1.1768, 0.9442, 1.6173],
        [5.8259, 5.0772, 4.5967,  ..., 1.5521, 1.2694, 1.5521]])

In [19]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output)

tensor([[6.1592e-01, 2.4733e-01, 5.8297e-03,  ..., 2.9385e-06, 1.5541e-06,
         2.6166e-06],
        [3.1591e-01, 1.0421e-01, 1.0149e-01,  ..., 8.3884e-06, 3.0501e-06,
         5.3005e-06],
        [6.9884e-01, 9.3563e-02, 2.2031e-02,  ..., 6.0585e-06, 1.2812e-05,
         1.2812e-05],
        ...,
        [4.3024e-01, 1.3894e-01, 3.6402e-02,  ..., 3.2580e-06, 1.7558e-05,
         1.5215e-05],
        [6.2014e-01, 1.3865e-01, 3.2559e-02,  ..., 3.0705e-06, 1.6317e-06,
         1.0167e-05],
        [7.1594e-01, 9.3552e-02, 2.5338e-02,  ..., 6.4501e-06, 2.9904e-06,
         6.4501e-06]], grad_fn=<SoftmaxBackward0>)


In [20]:
class Immunogenicity(nn.Module):
    def __init__(self, all_genes):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
        self.ig = nn.Parameter(torch.full((len(all_genes),), -1.0), requires_grad=True)
    
    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        ig = torch.sigmoid(self.ig[indices])
        return ig, filtered_genes

In [21]:
all_genes = pd.read_csv('./data/human_filtered.csv')
all_genes = all_genes['Gene'].values.tolist()

In [22]:
model = Immunogenicity(all_genes)

In [23]:
current_genes = gene_names

In [24]:
output, filtered_genes = model(list(current_genes[0]))

In [25]:
len(output)

99

In [68]:
class MIL(nn.Module):
    def __init__(self, all_genes):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes)
        self.alpha = nn.Parameter(torch.tensor(1.0))
        self.beta = nn.Parameter(torch.tensor(1.0))
    
    def forward(self, distances_list, gene_expressions_list, current_genes_list):
        bag_outputs = []
        
        # Since each bag may have different genes, we process them individually
        for distances, gene_expression, current_genes in zip(distances_list, gene_expressions_list, current_genes_list):
            # Process distances and gene expressions through their respective networks
            distances = self.distance(distances)  # Shape: [num_instances, 1]
            gene_expression = self.gene_expression(gene_expression)  # Shape: [num_instances, num_genes]

            # Get immunogenicity vector and filtered genes for the current bag
            immunogenicity_vector, filtered_genes = self.immunogenicity(current_genes)
            
            if len(filtered_genes) == 0:
                continue  # Skip this bag if no overlapping genes
            
            # Map gene names to indices
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            
            # Select the relevant gene expressions
            gene_expression = gene_expression[:, gene_indices]  # Shape: [num_instances, num_filtered_genes]
            
            # Compute z as the dot product between gene expression and immunogenicity
            z = gene_expression @ immunogenicity_vector  # Shape: [num_instances]
            z = z.unsqueeze(1)  # Shape: [num_instances, 1]
            
            # Compute the bag output
            bag_output = distances * z  # Element-wise multiplication
            bag_output = torch.sum(bag_output, dim=0)  # Sum over instances
            bag_output = torch.exp(self.alpha) * bag_output + self.beta
            #print(bag_output)
            #bag_output = torch.sigmoid(bag_output)  # Final output for the bag
            
            bag_outputs.append(bag_output)
        
        if len(bag_outputs) == 0:
            return None  # Handle this case appropriately in your training loop
        
        # Stack outputs for all bags in the batch
        return torch.stack(bag_outputs).squeeze(dim=1)  # Shape: [batch_size]

In [69]:
model = MIL(all_genes)

In [89]:
model.immunogenicity.ig

Parameter containing:
tensor([-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0004, -1.0000],
       requires_grad=True)

In [70]:
output = model(distances, gene_expressions, gene_names)
labels

tensor([1., 0.])

In [71]:
print(output)

tensor([1.0022, 1.0065], grad_fn=<SqueezeBackward1>)


In [72]:
labels[0]


tensor(1.)

In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [74]:
model = MIL(all_genes)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
model.to(device)

MIL(
  (distance): Distance(
    (sparsemax): Sparsemax()
    (softmax): Softmax(dim=0)
  )
  (gene_expression): Gene_expression(
    (softmax): Softmax(dim=-1)
  )
  (immunogenicity): Immunogenicity()
)

In [75]:
!pwd

/project/DPDS/Wang_lab/s439765/spatial_tcr/MIL_TCR


In [89]:
ig_scores_before_training = model.immunogenicity.ig
with open('ig_scores_before_training.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training'])
    for gene, score in zip(all_genes, ig_scores_before_training):
        writer.writerow([gene, score.item()])

In [76]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch

In [85]:

model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
num_epochs = 1000
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.1)

In [86]:
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm import tqdm


for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    # Lists to store outputs and labels for AUROC calculation
    all_outputs = []
    all_labels = []
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, batch_data in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")
    
            optimizer.zero_grad()
    
            # Unpack the batch data
            distances_list, gene_expressions_list, labels_list, core_idxs_list, gene_names_list, cell_ids_list = batch_data
            
            # Move data to device and prepare labels
            distances_list = [distances.to(device) for distances in distances_list]
            gene_expressions_list = [gene_exp.to(device) for gene_exp in gene_expressions_list]
            labels = torch.stack(labels_list).float().to(device)
            current_genes_list = gene_names_list  # List of gene names for each bag
            
            # Forward pass
            outputs = model(distances_list, gene_expressions_list, current_genes_list)
            
            if outputs is None:
                continue  # Skip this batch if the model returns None
            
            if outputs.shape[0] != labels.shape[0]:
                # Handle mismatch in batch sizes if necessary
                continue
            
            # Compute loss
            # Identify positive and negative outputs
            positive_idxs = (labels == 1).nonzero(as_tuple=True)[0]
            negative_idxs = (labels == 0).nonzero(as_tuple=True)[0]
            
            if positive_idxs.numel() == 0 or negative_idxs.numel() == 0:
                continue  # Skip batch if no positive or negative bags
            
            positive_output = outputs[positive_idxs]
            negative_outputs = outputs[negative_idxs]
            
            # Compute mean of negative outputs and loss
            mean_negative_output = negative_outputs.mean()
            positive_output = positive_output.mean()  # Should be a single value
            
            loss = torch.relu(mean_negative_output - positive_output)+0.1
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())
            
            # Accumulate outputs and labels for AUROC calculation
            all_outputs.extend(outputs.detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader)
    
    # Compute AUROC
    try:
        epoch_auc = roc_auc_score(all_labels, all_outputs)
    except ValueError:
        epoch_auc = float('nan')  # Handle case where AUROC can't be computed due to lack of positive/negative samples
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, AUROC: {epoch_auc:.4f}')
    
    # Validation loop
    model.eval()
    val_loss = 0.0
    val_all_outputs = []
    val_all_labels = []
    with torch.no_grad():
        with tqdm(val_dataloader, unit="batch") as vtepoch:
            for val_batch_data in vtepoch:
                # Unpack validation batch data
                val_distances_list, val_gene_expressions_list, val_labels_list, val_core_idxs_list, val_gene_names_list, val_cell_ids_list = val_batch_data
                
                # Move data to device and prepare labels
                val_distances_list = [distances.to(device) for distances in val_distances_list]
                val_gene_expressions_list = [gene_exp.to(device) for gene_exp in val_gene_expressions_list]
                val_labels = torch.stack(val_labels_list).float().to(device)
                val_current_genes_list = val_gene_names_list  # List of gene names for each bag
                
                # Forward pass
                val_outputs = model(val_distances_list, val_gene_expressions_list, val_current_genes_list)
                
                if val_outputs is None:
                    continue  # Skip this batch if the model returns None
                
                if val_outputs.shape[0] != val_labels.shape[0]:
                    # Handle mismatch in batch sizes if necessary
                    continue
                
                # Compute loss
                # Identify positive and negative outputs
                positive_idxs = (val_labels == 1).nonzero(as_tuple=True)[0]
                negative_idxs = (val_labels == 0).nonzero(as_tuple=True)[0]
                
                if positive_idxs.numel() == 0 or negative_idxs.numel() == 0:
                    continue  # Skip batch if no positive or negative bags
                
                positive_output = val_outputs[positive_idxs]
                negative_outputs = val_outputs[negative_idxs]
                
                # Compute mean of negative outputs and loss
                mean_negative_output = negative_outputs.mean()
                positive_output = positive_output.mean()  # Should be a single value
                
                loss = torch.relu(mean_negative_output - positive_output)
                val_loss += loss.item()
                vtepoch.set_postfix(val_loss=loss.item())
                
                # Accumulate outputs and labels for AUROC calculation
                val_all_outputs.extend(val_outputs.detach().cpu().numpy())
                val_all_labels.extend(val_labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        
        # Compute Validation AUROC
        try:
            val_epoch_auc = roc_auc_score(val_all_labels, val_all_outputs)
        except ValueError:
            val_epoch_auc = float('nan')  # Handle case where AUROC can't be computed
        
        print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_epoch_auc:.4f}')


Epoch 1/1000: 100%|██████████| 429/429 [00:01<00:00, 403.96batch/s, loss=0.1]  


Epoch [1/1000], Loss: 0.1035, AUROC: 0.3996


100%|██████████| 184/184 [00:00<00:00, 774.55batch/s, val_loss=0.00355] 


Validation Loss: 0.0044, Validation AUROC: 0.3588


Epoch 2/1000: 100%|██████████| 429/429 [00:01<00:00, 393.14batch/s, loss=0.105]


...

Validation Loss: 0.0004, Validation AUROC: 0.3778


Epoch 716/1000: 100%|██████████| 429/429 [00:01<00:00, 407.44batch/s, loss=0.1]  


Epoch [716/1000], Loss: 0.1002, AUROC: 0.3917


 10%|█         | 19/184 [00:00<00:00, 780.52batch/s, val_loss=0.00041]


KeyboardInterrupt: 

In [48]:
def predict(model, adata, device, radius=200, max_instances=None):
    model.eval()
    
    dataset = BagsDataset(adata, immune_cell='tcell', radius=radius, max_instances=max_instances)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=custom_collate_fn)
    
    predictions = np.full(len(adata.obs), np.nan)  # Initialize predictions array with NaNs

    with torch.no_grad():
        with tqdm(dataloader, unit="batch") as tepoch:
            for distances, gene_expressions, _, core_idx, current_genes in tepoch:
                tepoch.set_description("Predicting")
                
                # Move data to the device
                distances = [d.to(device) for d in distances]
                gene_expressions = [g.to(device) for g in gene_expressions]
                
                output = model(distances, gene_expressions, list(current_genes[0]))
                
                # Ensure we extract a single element from core_idx and output before using them
                predictions[int(core_idx.item())] = output.cpu().numpy().flatten().item()

    adata.obs['tcr_predict'] = predictions
    return adata

In [49]:
adata = sc.read_h5ad('../test.h5ad')

In [50]:
adata = predict(model, adata,radius=200, device=device)

Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:00<00:00, 6262.95it/s]


Total bags created: 3858
Average instances per bag: 9


Predicting: 100%|██████████| 3858/3858 [00:04<00:00, 846.80batch/s]


In [51]:
adata.obs

,X,Y,T,cell_type,tcr_predict
AACACGTGCATCGCAC-1,7600,2200,0,0,0.505103
AACACTTGGCAAGGAA-1,4700,7100,1,1,0.557644
AACAGGATTCATAGTT-1,4900,4300,1,0,0.717114
AACAGGTTATTGCACC-1,2800,8600,0,0,0.500146
AACAGGTTCACCGAAG-1,5100,4100,1,0,0.500000
...,...,...,...,...,...
TGTTGGAACCTTCCGC-1,3500,3500,0,0,0.516184
TGTTGGAACGAGGTCA-1,2800,7200,0,1,0.544874
TGTTGGAAGCTCGGTA-1,100,9500,0,0,0.721982
TGTTGGATGGACTTCT-1,1300,5300,0,0,0.512577


In [46]:
print(torch.sigmoid(model.immunogenicity.ig))

tensor([0.2342, 0.2457, 0.2458,  ..., 0.2689, 0.2689, 0.2689],
       grad_fn=<SigmoidBackward0>)


In [44]:
ig_scores_after_training = model.immunogenicity.ig
with open('ig_scores_after_training.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score After Training'])
    for gene, score in zip(all_genes, ig_scores_after_training):
        writer.writerow([gene, score.item()])
        
with open('ig_score_changes.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training', 'IG Score After Training'])
    for gene, before, after in zip(all_genes, ig_scores_before_training, ig_scores_after_training):
        writer.writerow([gene, before.item(), after.item()])